# Task 1 — Dataset Understanding & Exploratory Analysis
**COMP41840 AI for Health**  
**Owner:** Sergio  
**Dataset:** Multi-Modal Breast Cancer Dataset (780 patients)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_theme(style='whitegrid')

DATA_ROOT = Path('../data/raw')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)

_SRC = Path('../src').resolve()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from patient_split import kaggle_multimodal_available, build_aligned_manifest

USE_ALIGNED = kaggle_multimodal_available(DATA_ROOT)
CLASSES = ['benign', 'malignant', 'normal']


def iter_class_images(data_root: Path, cls: str):
    paths = []
    d = data_root / cls / 'images'
    if d.exists():
        paths.extend(sorted(d.glob('*.png')))
    for sub in sorted((data_root / 'dataset').glob(f'dataset*/{cls}/images')):
        paths.extend(sorted(sub.glob('*.png')))
    return paths


def first_class_image(data_root: Path, cls: str):
    paths = iter_class_images(data_root, cls)
    return paths[0] if paths else None

## 1.1 — Image Dataset Structure

In [ ]:
# Count images per class (flat or nested Kaggle layout)
counts = {cls: len(iter_class_images(DATA_ROOT, cls)) for cls in CLASSES}
for cls, n in counts.items():
    print(f'{cls}: {n} images')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.keys(), counts.values(), color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_title('Class Distribution (Images)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/class_distribution.png', dpi=150)
plt.show()

## 1.2 — Image Resolution & Format

In [ ]:
widths, heights, class_labels = [], [], []
for cls in CLASSES:
    paths = iter_class_images(DATA_ROOT, cls)[:50]
    for p in paths:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            class_labels.append(cls)

size_df = pd.DataFrame({'width': widths, 'height': heights, 'class': class_labels})
if len(size_df):
    print(size_df.groupby('class')[['width', 'height']].describe())
else:
    print('No images found for resolution sampling.')

## 1.3 — Sample Images per Class

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
fig.suptitle('Sample Ultrasound Images per Class', fontsize=14)

for row, cls in enumerate(CLASSES):
    samples = iter_class_images(DATA_ROOT, cls)[:3]
    for col, p in enumerate(samples):
        axes[row][col].imshow(Image.open(p), cmap='gray')
        axes[row][col].set_title(f'{cls} — {p.name}', fontsize=8)
        axes[row][col].axis('off')
    for col in range(len(samples), 3):
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/sample_images.png', dpi=150)
plt.show()

## 1.4 — Mask Overlay Visualisation

In [ ]:
def mask_path_for_image(img_path: Path) -> Path | None:
    p = Path(img_path)
    m = p.parent.parent / 'masks' / p.name
    return m if m.exists() else None


examples = []
for cls in ('malignant', 'benign'):
    for p in iter_class_images(DATA_ROOT, cls)[:5]:
        m = mask_path_for_image(p)
        if m is not None:
            examples.append((p, m, cls))
            break

if not examples:
    print('No image/mask pairs found (check dataset layout).')
else:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    img_p, mk_p, cls = examples[0]
    raw = np.array(Image.open(img_p).convert('RGB'))
    msk = np.array(Image.open(mk_p).convert('L'))
    msk = (msk > 127).astype(float)
    if msk.shape[:2] != raw.shape[:2]:
        msk = np.array(Image.fromarray((msk * 255).astype(np.uint8)).resize((raw.shape[1], raw.shape[0])))
        msk = (msk > 127).astype(float)
    overlay = raw.copy()
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] * 0.6 + msk * 255 * 0.4, 0, 255)
    axes[0].imshow(raw)
    axes[0].set_title(f'Image ({cls})')
    axes[1].imshow(msk, cmap='gray')
    axes[1].set_title('Mask')
    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title('Overlay')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'Mask alignment — {img_p.name}')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures/mask_overlay_example.png', dpi=150)
    plt.show()

## 1.5 — Tabular / Clinical Data

In [ ]:
clinical_path = DATA_ROOT.parent / 'clinical.csv'

if USE_ALIGNED:
    df = build_aligned_manifest(DATA_ROOT)
    print('Loaded: merged Kaggle tabular + image paths (647 patients with ultrasound)')
    print('Shape:', df.shape)
    print(df.dtypes.head(20))
    print(df.head())
else:
    if not clinical_path.exists():
        raise FileNotFoundError(
            f'Missing {clinical_path}. Extract Kaggle data under data/raw/ or add clinical.csv.'
        )
    df = pd.read_csv(clinical_path)
    print('Loaded:', clinical_path)
    print('Shape:', df.shape)
    print(df.dtypes)
    print(df.head())

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.concat([missing, missing_pct], axis=1, keys=['count', '%'])
print(miss_df.head(25))
if len(miss_df) > 25:
    print('...')

In [ ]:
if USE_ALIGNED and 'class' in df.columns:
    vc = df['class'].value_counts()
    print(vc)
    vc.plot(kind='bar', title='Tabular class (benign / malignant)')
elif 'label' in df.columns:
    print(df['label'].value_counts())
    df['label'].value_counts().plot(kind='bar', title='Tabular Label Distribution')
elif 'Event' in df.columns:
    print(df['Event'].value_counts())
    df['Event'].astype(str).value_counts().plot(kind='bar', title='Event distribution')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/tabular_label_distribution.png', dpi=150)
plt.show()

In [ ]:
drop_vis = {'patient_id', 'label_enc', 'image_path', 'split', 'Patient ID'}
num_candidates = [c for c in df.select_dtypes(include='number').columns if c not in drop_vis]
numerical_cols = num_candidates[:12]
if numerical_cols:
    df[numerical_cols].hist(figsize=(14, 10), bins=20)
    plt.suptitle('Numerical Feature Distributions (sample of columns)')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures/feature_distributions.png', dpi=150)
    plt.show()
else:
    print('No numeric columns to plot.')

In [ ]:
hm_cols = num_candidates[:15]
if len(hm_cols) >= 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(df[hm_cols].corr(), annot=len(hm_cols) <= 10, fmt='.2f', cmap='coolwarm', ax=ax)
    ax.set_title('Feature correlation (subset)')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures/correlation_heatmap.png', dpi=150)
    plt.show()
else:
    print('Not enough numeric columns for correlation heatmap.')

## 1.6 — Summary

- **Images:** Per-class counts are in section 1.1. Paths support both `data/raw/<class>/images/` and nested `data/raw/dataset/dataset*/<class>/images/`.
- **Tabular:** With the Kaggle extract, section 1.5 uses the merged clinical + molecular table (647 patients with matching ultrasound). Otherwise it uses `data/clinical.csv`.
- **Masks:** Example overlay saved as `results/figures/mask_overlay_example.png` when mask files exist.
- **Imbalance:** Benign cases typically exceed malignant; Tasks 2–3 use class weighting (`CrossEntropyLoss` weights, `scale_pos_weight` in XGBoost).

Re-run section 1.1 after data setup to refresh counts for the table in your report.